# Topic: SQL: Daily Active Users (DAU) & Stickiness

## Definition
*   **Daily Active Users (DAU):** The number of unique users who perform at least one qualifying action (e.g., login, click, purchase) on a product in a single day.
*   **DAU/MAU Ratio (Stickiness):** The percentage of monthly active users who engage with the product on a daily basis.

## Why Interviewers Ask This
*   **Fundamental Product Metric:** DAU and stickiness are universally used KPIs across consumer tech, SaaS, and gaming to measure product health.
*   **SQL Fundamentals Test:** It cleanly tests a candidate's grasp of `COUNT(DISTINCT)`, grouping by dates, and window functions (rolling averages).
*   **Business Acumen:** Interviewers want to see if you understand the *meaning* behind the metrics, not just the math (e.g., recognizing that 50% stickiness is excellent).

## Core Concepts
*   **Unique Counting:** One user logging in 10 times in one day is exactly 1 DAU. `COUNT(DISTINCT user_id)` is mandatory.
*   **Time Aggregation (WAU/MAU):** Grouping user activity into weekly or monthly buckets to measure broader engagement trends.
*   **Rolling Averages:** Using window functions to smooth out day-to-day volatility (like weekend dips) to reveal actual trends.

## When to Use
*   Whenever a prompt asks for "unique daily engagement," "active users," or "stickiness."
*   To evaluate the success of a new feature launch (did daily engagement spike and stay high?).
*   To compare product usage across different platforms (e.g., iOS vs. Android DAU).

## Advantages
*   **Standardized Benchmarking:** Stickiness (DAU/MAU) provides a standardized way to compare engagement across different products or features.
*   **High Sensitivity:** DAU reacts immediately to outages, marketing campaigns, or viral events.
*   **Actionable:** Rolling DAU trends clearly indicate whether product growth is accelerating or decaying.

## Limitations
*   **Susceptible to Noise:** Raw DAU fluctuates wildly based on the day of the week or minor external events.
*   **Lacks Depth:** DAU tells you *who* showed up, but not *what* they did or how long they stayed (Time Spent is often needed alongside it).
*   **Missing Dates Trap:** Days with zero activity will simply not appear in standard SQL `GROUP BY` output.

## Common Comparisons
*   **DAU vs. Sessions:** DAU counts the unique *human/account*; Sessions count the number of distinct *visits*.
*   **Raw DAU vs. Rolling DAU:** Raw DAU shows daily reality (good for ops); Rolling DAU shows the underlying trend (good for strategy).
*   **DAU/MAU vs. Churn:** Stickiness measures how often retained users engage; Churn measures how many users leave the product entirely.

## Common Interview Traps
*   **Using COUNT(*):** Counting events instead of unique users will drastically inflate your numbers.
*   **Including Bots/Test Data:** Forgetting to filter out internal employees or automated bots (`WHERE is_bot = false`).
*   **Denominator Mismatch in DAU/MAU:** Calculating DAU for a specific day, but dividing by the MAU of a different time period.
*   **Division by Integer:** In many SQL dialects, dividing two integers yields an integer (0). Always multiply by 100.0 first.

## MySQL Syntax
*   **DAU Calculation:** `COUNT(DISTINCT user_id)`
*   **7-Day Rolling Avg DAU:** `AVG(dau) OVER (ORDER BY event_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)`
*   **Month Truncation (MySQL):** `DATE_FORMAT(event_date, '%Y-%m-01')`

## Important Formula
*   **Stickiness Ratio:** `(DAU / MAU) * 100` 
    *   *Note: For a monthly aggregation, it is usually calculated as: (Average DAU for the Month / MAU for the Month) * 100.*

## 45-Second Interview Answer
"To calculate DAU, I group the data by date and use `COUNT(DISTINCT user_id)`. It is critical to use DISTINCT so highly active users don't inflate the metric. To calculate stickiness, which is the DAU/MAU ratio, I typically use two CTEs: one for average daily users in a month, and one for total unique monthly users, then join them to calculate the percentage. Because DAU is noisy, I frequently apply a 7-day rolling average using a window function to smooth out weekend seasonality."

## Example Questions:

### Q1. Calculate WAU (Weekly Active Users) for each week of 2025.

**Ideal Answer (MySQL):**
```sql
SELECT 
    DATE_ADD(event_date, INTERVAL -WEEKDAY(event_date) DAY) AS event_week,
    COUNT(DISTINCT user_id) AS wau
FROM user_events
WHERE event_date >= '2025-01-01' AND event_date < '2026-01-01'
GROUP BY event_week
ORDER BY event_week;
```
*   **Common Mistakes:** Using `COUNT(*)` instead of `COUNT(DISTINCT user_id)`. Using `WEEK(event_date)` which can format inconsistently across years instead of creating an actual start-of-week date object.
*   **Interviewer Follow-up:** "How would you handle a user who logged in on Sunday night and continued their session into Monday morning?" (Answer: Define a strict timezone cutoff for the business, e.g., midnight UTC, and assign the event timestamp to that boundary).

### Q2. Find the day with the highest DAU in Q1 2025.

**Ideal Answer (MySQL):**
```sql
SELECT 
    event_date,
    COUNT(DISTINCT user_id) AS dau
FROM user_events
WHERE event_date >= '2025-01-01' AND event_date <= '2025-03-31'
GROUP BY event_date
ORDER BY dau DESC
LIMIT 1;
```
*   **Common Mistakes:** Not filtering for Q1 dates specifically, or ordering ascending instead of descending.
*   **Interviewer Follow-up:** "What if there is a tie for the highest DAU?" (Answer: `LIMIT 1` will arbitrarily pick one. To show all ties, I would use the `RANK()` window function over the counts and filter `WHERE rank = 1` in an outer query).

### Q3. Calculate the DAU/MAU ratio for each month in 2025.

**Ideal Answer (MySQL):**
```sql
WITH daily_active AS (
    SELECT 
        event_date,
        DATE_FORMAT(event_date, '%Y-%m-01') AS event_month,
        COUNT(DISTINCT user_id) AS dau
    FROM user_events
    WHERE YEAR(event_date) = 2025
    GROUP BY event_date, event_month
),
monthly_avg_dau AS (
    SELECT 
        event_month,
        AVG(dau) AS avg_daily_users
    FROM daily_active
    GROUP BY event_month
),
monthly_active AS (
    SELECT 
        DATE_FORMAT(event_date, '%Y-%m-01') AS event_month,
        COUNT(DISTINCT user_id) AS mau
    FROM user_events
    WHERE YEAR(event_date) = 2025
    GROUP BY event_month
)
SELECT 
    m.event_month,
    m.mau,
    ROUND(d.avg_daily_users, 1) AS avg_dau,
    ROUND((d.avg_daily_users / m.mau) * 100.0, 2) AS stickiness_ratio
FROM monthly_active m
JOIN monthly_avg_dau d ON m.event_month = d.event_month
ORDER BY m.event_month;
```
*   **Common Mistakes:** Attempting to divide a single day's DAU by the entire month's MAU without averaging the DAU for that specific month first. 
*   **Interviewer Follow-up:** "Why is calculating Average DAU first, then dividing by MAU, better than finding the stickiness ratio for each day and then averaging the percentages?" (Answer: Mathematically, averaging percentages can skew the results if the denominators (MAU) fluctuate wildly; aggregating the absolutes first is more statistically sound).

### Q4. Find the 7-day rolling average DAU for each platform (iOS, Android, Web).

**Ideal Answer (MySQL):**
```sql
WITH daily_platform_dau AS (
    SELECT 
        event_date,
        platform,
        COUNT(DISTINCT user_id) AS dau
    FROM user_events
    GROUP BY event_date, platform
)
SELECT 
    event_date,
    platform,
    dau,
    ROUND(AVG(dau) OVER (
        PARTITION BY platform 
        ORDER BY event_date 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ), 1) AS rolling_7day_dau
FROM daily_platform_dau
ORDER BY platform, event_date;
```
*   **Common Mistakes:** Forgetting the `PARTITION BY platform` in the window function, which causes the rolling average to bleed across different platforms.
*   **Interviewer Follow-up:** "For the first 6 days in the dataset, the rolling average computes over fewer than 7 days. How would you force it to output NULL until a full 7 days of data is available?" (Answer: By adding a `COUNT(dau) OVER (...)` in the window function and wrapping the calculation in a `CASE WHEN count = 7 THEN ... ELSE NULL END`).

### Q5. Identify days where DAU dropped more than 20% from the previous day.

**Ideal Answer (MySQL):**
```sql
WITH daily_dau AS (
    SELECT 
        event_date,
        COUNT(DISTINCT user_id) AS dau
    FROM user_events
    GROUP BY event_date
),
lagged_dau AS (
    SELECT 
        event_date,
        dau,
        LAG(dau) OVER (ORDER BY event_date) AS prev_dau
    FROM daily_dau
)
SELECT 
    event_date,
    dau,
    prev_dau,
    ROUND(100.0 * (dau - prev_dau) / prev_dau, 2) AS pct_change
FROM lagged_dau
WHERE (dau - prev_dau) / prev_dau <= -0.20
ORDER BY event_date;
```
*   **Common Mistakes:** Calculating the percentage change incorrectly (e.g., `prev_dau / dau`), or forgetting to handle potential division-by-zero if `prev_dau` can be 0.
*   **Interviewer Follow-up:** "If a day drops by 20%, but it's a Sunday and every Sunday drops by 20%, this alert is just noise. How could you adjust the query to only find anomalies?" (Answer: Instead of comparing to the previous day, I would use `LAG(dau, 7)` to compare against the same day of the previous week).

## Practice Questions:

### Q1: E-Commerce Stickiness & Metric Interpretation
**Scenario:** Calculate the Stickiness Ratio (Avg Daily Customers / Monthly Active Customers) for November 2004 based on placed orders. Then, explain why comparing this ratio to a social media benchmark (like 50%) is flawed.

**Answer:**
```sql
WITH dau AS (
    SELECT 
        orderDate AS event_date,
        DATE_FORMAT(orderDate, '%Y-%m') AS event_month,
        COUNT(DISTINCT customerNumber) AS dailyCustomerCount
    FROM orders
    WHERE DATE_FORMAT(orderDate, '%Y-%m') = '2004-11'
    GROUP BY orderDate, event_month
),
monthly_avg_dau AS (
    SELECT 
        event_month,
        AVG(dailyCustomerCount) AS monthlyAvgDAU
    FROM dau
    GROUP BY event_month
),
mau AS (
    SELECT
        DATE_FORMAT(orderDate, '%Y-%m') AS event_month,
        COUNT(DISTINCT customerNumber) AS MAU
    FROM orders
    WHERE DATE_FORMAT(orderDate, '%Y-%m') = '2004-11'
    GROUP BY event_month
)
SELECT 
    m.event_month, 
    ROUND(mad.monthlyAvgDAU, 2) AS monthlyAvgDAU, 
    m.MAU,
    ROUND((100.0 * mad.monthlyAvgDAU) / m.MAU, 2) AS stickiness_pct
FROM mau m
JOIN monthly_avg_dau mad ON m.event_month = mad.event_month;
```

**Answer:**
"Comparing e-commerce purchase stickiness to social media stickiness is flawed because the natural frequency of the two products is completely different. A user might check WhatsApp 10 times a day, but naturally only buys furniture once a year. To measure true engagement for e-commerce, I would recommend redefining an 'active' event as top-of-funnel actions (app opens, item views, adding to cart) rather than completed purchases. Alternatively, we should change the time horizon and measure 'Repeat Purchase Rate' over a year."
